In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Pertemuan10-StructuredStreaming") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")


spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

26/11/11 20:08:42 WARN Utils: Your hostname, ubuntu resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/11 20:08:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 20:08:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
folder_stream = "simulasi_stream_transaksi" ### folder ini akan menyimpa nama folder simulasi_stream balbal nanti buat menyimpan file csv yang bakal di baca spark
folder_checkpoint = "checkpoint_pertemuan10"

shutil.rmtree(folder_stream, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

print(f"Folder simulasi streaming '{folder_stream}' siap (kosong).")

### 

Folder simulasi streaming 'simulasi_stream_transaksi' siap (kosong).


In [3]:
skema_transaksi = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
])

print("Skema berhasil didefinisikan:")
print(skema_transaksi)

Skema berhasil didefinisikan:
StructType([StructField('trx_id', StringType(), True), StructField('kota', StringType(), True), StructField('kategori', StringType(), True), StructField('pendapatan', IntegerType(), True)])


In [5]:
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_transaksi) \
    .option("header", True) \
    .load(folder_stream)

print("Streaming DataFrame berhasil didefinisikan (belum berjalan — baru 'rencana').")
print("df_stream.isStreaming:", df_stream.isStreaming)

Streaming DataFrame berhasil didefinisikan (belum berjalan — baru 'rencana').
df_stream.isStreaming: True


In [6]:
agregasi_stream = df_stream.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan"),
    count("trx_id").alias("jumlah_transaksi")
)

print("Transformasi berhasil didefinisikan.")

Transformasi berhasil didefinisikan.


In [7]:
query = agregasi_stream.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hasil_agregasi") \
    .start()

print("Streaming query dimulai! Status:", query.status)

Streaming query dimulai! Status: {'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}


In [8]:

pd.DataFrame({
    "trx_id": ["T1", "T2", "T3"],
    "kota": ["Magelang", "Solo", "Magelang"],
    "kategori": ["Elektronik", "Fashion", "Makanan"],
    "pendapatan": [150000, 200000, 75000],
}).to_csv(f"{folder_stream}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(6) 

spark.sql("SELECT * FROM hasil_agregasi ORDER BY total_pendapatan DESC").show()

Batch 1 dikirim. Menunggu Spark memproses...


+--------+----------------+----------------+
|    kota|total_pendapatan|jumlah_transaksi|
+--------+----------------+----------------+
|Magelang|          225000|               2|
|    Solo|          200000|               1|
+--------+----------------+----------------+



In [9]:
pd.DataFrame({
    "trx_id": ["T4", "T5"],
    "kota": ["Magelang", "Yogyakarta"],
    "kategori": ["Fashion", "Elektronik"],
    "pendapatan": [300000, 500000],
}).to_csv(f"{folder_stream}/batch_2.csv", index=False)

print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(6)

spark.sql("SELECT * FROM hasil_agregasi ORDER BY total_pendapatan DESC").show()

Batch 2 dikirim. Menunggu Spark memproses...
+----------+----------------+----------------+
|      kota|total_pendapatan|jumlah_transaksi|
+----------+----------------+----------------+
|  Magelang|          525000|               3|
|Yogyakarta|          500000|               1|
|      Solo|          200000|               1|
+----------+----------------+----------------+



In [11]:
query.stop()
print("Streaming query dihentikan. Status aktif:", query.isActive)

Streaming query dihentikan. Status aktif: False


In [12]:
skema_dengan_waktu = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
    StructField("waktu_transaksi", TimestampType(), True),
])

# Membuat ulang folder simulasi (kosong) untuk demo window waktu
folder_stream2 = "simulasi_stream_waktu"
shutil.rmtree(folder_stream2, ignore_errors=True)
os.makedirs(folder_stream2, exist_ok=True)

df_stream2 = spark.readStream \
    .format("csv") \
    .schema(skema_dengan_waktu) \
    .option("header", True) \
    .load(folder_stream2)

# Mengelompokkan berdasarkan window waktu 1 menit, dihitung dari kolom waktu_transaksi
agregasi_window = df_stream2.groupBy(
    window(col("waktu_transaksi"), "1 minute")
).agg(
    spark_sum("pendapatan").alias("total_pendapatan"),
    count("trx_id").alias("jumlah_transaksi")
)

query2 = agregasi_window.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hasil_window") \
    .start()

print("Streaming query window waktu dimulai.")

Streaming query window waktu dimulai.


In [13]:
from datetime import datetime

# Mengirim data dengan timestamp saat ini
waktu_sekarang = datetime.now()
pd.DataFrame({
    "trx_id": ["W1", "W2"],
    "kota": ["Semarang", "Solo"],
    "pendapatan": [120000, 180000],
    "waktu_transaksi": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2,
}).to_csv(f"{folder_stream2}/batch_w1.csv", index=False)

time.sleep(6)
spark.sql("SELECT window.start, window.end, total_pendapatan, jumlah_transaksi FROM hasil_window").show(truncate=False)

+-------------------+-------------------+----------------+----------------+
|start              |end                |total_pendapatan|jumlah_transaksi|
+-------------------+-------------------+----------------+----------------+
|2026-11-11 20:10:00|2026-11-11 20:11:00|300000          |2               |
+-------------------+-------------------+----------------+----------------+



In [38]:
query2.stop()
spark.stop()
print("Seluruh streaming query dihentikan, SparkSession ditutup.")


Seluruh streaming query dihentikan, SparkSession ditutup.


In [ ]:
### LATIHAN MANDIRI

In [39]:
### LATIHAN MANDIRI

# Membuat SparkSession
spark = SparkSession.builder.appName("Latihan10").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Mendefinisikan folder streaming
folder_latihan = "latihan10_stream"

# Mendefinisikan skema transaksi
skema_latihan = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
])
print("Siap untuk latihan. Folder:", folder_latihan)

Siap untuk latihan. Folder: latihan10_stream


In [51]:
from pyspark.sql.functions import avg

df_latihan = spark.readStream \
    .format("csv") \
    .schema(skema_latihan) \
    .option("header", True) \
    .load(folder_latihan) 

df_agg = df_latihan \
    .groupBy("kategori") \
    .agg(avg("pendapatan").alias("rata_rata_pendapatan"))

query = df_agg.writeStream \
    .format("memory") \
    .queryName("Latihan_agg") \
    .outputMode("complete") \
    .start()


In [52]:
query.processAllAvailable()

In [53]:
data = """trx_id, kategori,pendapatan
TR102,Makanan,50000
TR104,Makanan,65000
TR103,Minuman,20000
TR105,Minuman,30000
TR111,Elektronik,1200000
TR113,Elektronik,500000
TR115,Elektronik,2500000
TR120,Langganan,150000
TR122,Langganan,770000
"""

with open(os.path.join(folder_latihan, "transaksi.csv"), "w") as file:
    file.write(data)


In [54]:
for q in spark.streams.active:
    print("Nama:", q.name)
    print("Status:", q.isActive)

Nama: Latihan_agg
Status: True


In [55]:
q = spark.streams.active[0]
q.processAllAvailable()

In [56]:
spark.sql("SELECT * FROM Latihan_agg").show()

+----------+--------------------+
|  kategori|rata_rata_pendapatan|
+----------+--------------------+
|   Minuman|             25000.0|
|   Makanan|             57500.0|
| Langganan|            460000.0|
|Elektronik|           1400000.0|
+----------+--------------------+



In [46]:
for q in spark.streams.active:
    q.stop()

In [57]:
### LATIHAN MANDIRI SOAL 2

# dengan 5 databaru dgn 5 transaksi tambahan

data_baru = """trx_id,kategori,pendapatan
TR130,Makanan,10000
TR131,Minuman,100000
TR135,Elektronik,10000000
TR134,Makanan,16000
TR137,Minuman,110000
"""

with open(os.path.join(folder_latihan, "transaksi_baru.csv"), "w") as file: # with open untuk membuat file csv baru.
    file.write(data_baru) # os.path untuk menetukan lokasi file
    # file.write untuk menulis data ke file

print("File transaksi_baru.csv berhasil dibuat.")

File transaksi_baru.csv berhasil dibuat.


In [58]:
import time

time.sleep(5) # spark jadi punya waktu untuk mendeteksi file baru

query.processAllAvailable() # memastikan data semua sudah tersedia

In [60]:
spark.sql("SELECT * FROM latihan_agg").show()

+----------+--------------------+
|  kategori|rata_rata_pendapatan|
+----------+--------------------+
|   Minuman|             65000.0|
|   Makanan|             35250.0|
| Langganan|            460000.0|
|Elektronik|           3550000.0|
+----------+--------------------+



In [76]:
### LATIHAN MANDIRI SOAL 3

# Kirim batch kedua dengan kategori sama

# data transaksi baru
data_batch3 = """trx_id,kategori,pendapatan 
TR150,Makanan,80000
TR151,Minuman,40000
TR152,Elektronik,2000000
TR153,Langganan,300000
"""

with open(os.path.join(folder_latihan, "transaksi_batch3.csv"), "w") as file:
    file.write(data_batch3)

In [77]:
import time 

time.sleep(5)

query.processAllAvailable()

In [78]:
spark.sql("SELECT * FROM latihan_agg").show()

+----------+--------------------+
|  kategori|rata_rata_pendapatan|
+----------+--------------------+
|   Minuman|             60000.0|
|   Makanan|             44200.0|
| Langganan|   406666.6666666667|
|Elektronik|           3240000.0|
+----------+--------------------+



### LATIHAN MANDIRI SOAL 4

Karena data Streaming itu terus masuk setiap saat terus satu-persatu sehingga Spark belum tahu tipe datanya keseluruhan, dia harus memeriksa satu-persatu struktur dan tipe datanya dari awal.
Sedangkan saat batch inferSchema=True itu digunakan karena datanya sudah ada dalam batch atau kumpulan gitu sehingga Spark bisa mendeteksi tipe datanya secara otomatis, seperti langung di scan satu folder jadi langsung ketahuan semua.